# Transformer Foundations, Part 4B: Decoder-Only PyTorch Lab

Part 4A ended with a concrete implementation contract: variable-length sentences must become aligned inputs and targets; causal masking, padding masking, and ignored targets must remain distinct; generation must stop safely; and a cache must preserve logits.

This lab resumes Riverside's `aria / meridian / signal` example in a fresh kernel. It builds one inspectable decoder, not another complete Transformer course.

| Build step | Artifact | Check |
|---|---|---|
| 1 | `Dataset`, `DataLoader`, dynamic collator | dtypes, shapes, and mask meanings |
| 2 | cached causal attention block | full-sequence logits and cache state |
| 3 | flattened token loss | `(B,T,V) -> (B*T,V)` alignment |
| 4 | manual optimization loop | measured loss movement |
| 5 | greedy and sampled generation | EOS and length-limit behavior |
| 6 | cached generation | next-logit parity with recomputation |

**Prerequisite recap:** Part 3 owns multi-head attention, FFNs, residuals, and normalization. The code below only adds the causal-language-model boundaries around them.


In [ ]:
# Imports and deterministic lab state
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence

SEED = 404
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")


## 1. Dataset items stay natural length; the collator builds rectangles

A `Dataset` answers "what is one example?" A `DataLoader` decides batching and shuffle order. The collator owns the batch-only problem: examples have different lengths, but a tensor batch must be rectangular.

Dynamic padding stops at the longest sentence in this batch. It produces three aligned objects:

- integer input IDs for `nn.Embedding`;
- integer next-token targets, using `-100` only where no lesson exists;
- a Boolean key-padding mask, where `True` means attention must not read that key.


In [ ]:
# A small sentence Dataset with shifted next-token targets
corpus = [
    "aria heard the signal",
    "aria followed the signal",
    "meridian carried the signal",
    "the signal crossed meridian",
    "aria returned to meridian",
    "meridian heard aria",
] * 24
special = ["<pad>", "<bos>", "<eos>"]
vocabulary = special + sorted({word for text in corpus for word in text.split()})
token_to_id = {token: index for index, token in enumerate(vocabulary)}
id_to_token = {index: token for token, index in token_to_id.items()}
PAD_ID, BOS_ID, EOS_ID = (token_to_id[token] for token in special)

class RiversideDataset(Dataset):
    def __init__(self, texts):
        self.examples = []
        for text in texts:
            ids = [BOS_ID] + [token_to_id[word] for word in text.split()] + [EOS_ID]
            values = torch.tensor(ids, dtype=torch.long)
            self.examples.append((values[:-1], values[1:]))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


In [ ]:
# Dynamic collation: pad inputs, ignore padded targets, and mark padded keys
def collate_batch(examples):
    inputs, targets = zip(*examples)
    input_ids = pad_sequence(inputs, batch_first=True, padding_value=PAD_ID)
    labels = pad_sequence(targets, batch_first=True, padding_value=-100)
    key_padding_mask = input_ids.eq(PAD_ID)
    return input_ids, labels, key_padding_mask

loader = DataLoader(
    RiversideDataset(corpus),
    batch_size=12,
    shuffle=True,
    collate_fn=collate_batch,
    generator=torch.Generator().manual_seed(SEED),
)
input_ids, labels, key_padding_mask = next(iter(loader))
print("input_ids:", tuple(input_ids.shape), input_ids.dtype)
print("labels:", tuple(labels.shape), labels.dtype)
print("key_padding_mask:", tuple(key_padding_mask.shape), key_padding_mask.dtype)
print("ignored target rows:", int(labels.eq(-100).sum()))
assert input_ids.shape == labels.shape == key_padding_mask.shape


#### Walkthrough: the three mask jobs remain separate

The collator has created only batch-specific padding information. The model will create its causal triangle from sequence length. Cross-entropy will use `-100` labels to skip rows. None of those three operations substitutes for either of the others.

## 2. Add a cache without hiding the attention mechanics

A cache changes the attention input shape, not the attention rule. During full training, query and key lengths are both `T` and the causal triangle is active. During cached generation, the query length is `1`; its keys are the stored prefix plus that new position.


In [ ]:
# Multi-head causal self-attention with an optional per-layer KV cache
class CachedCausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        if d_model % n_heads:
            raise ValueError("d_model must be divisible by n_heads")
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.output = nn.Linear(d_model, d_model)

    def forward(self, x, key_padding_mask=None, cache=None):
        batch, query_length, width = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        reshape = lambda value: value.view(
            batch, query_length, self.n_heads, self.head_dim
        ).transpose(1, 2)
        q, k, v = map(reshape, (q, k, v))
        if cache is not None:
            k = torch.cat([cache[0], k], dim=2)
            v = torch.cat([cache[1], v], dim=2)

        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        key_length = k.size(2)
        past_length = key_length - query_length
        query_positions = torch.arange(query_length, device=x.device) + past_length
        key_positions = torch.arange(key_length, device=x.device)
        causal_mask = key_positions[None, :] > query_positions[:, None]
        scores = scores.masked_fill(causal_mask, float("-inf"))
        if key_padding_mask is not None:
            scores = scores.masked_fill(
                key_padding_mask[:, None, None, :], float("-inf")
            )
        weights = torch.softmax(scores, dim=-1)
        mixed = weights @ v
        mixed = mixed.transpose(1, 2).contiguous().view(batch, query_length, width)
        return self.output(mixed), (k, v), weights


In [ ]:
# Pre-normalized block: the Part 3 mechanism plus cache plumbing
class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attention_norm = nn.LayerNorm(d_model)
        self.attention = CachedCausalSelfAttention(d_model, n_heads)
        self.ffn_norm = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x, key_padding_mask=None, cache=None):
        update, new_cache, weights = self.attention(
            self.attention_norm(x), key_padding_mask, cache
        )
        x = x + update
        x = x + self.ffn(self.ffn_norm(x))
        return x, new_cache, weights


In [ ]:
# Token/position embeddings, one decoder block, and vocabulary logits
class TinyCausalLM(nn.Module):
    def __init__(self, vocab_size, max_length=24, d_model=32):
        super().__init__()
        self.max_length = max_length
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.position_embedding = nn.Embedding(max_length, d_model)
        self.blocks = nn.ModuleList([DecoderBlock(d_model, 4, 64)])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids, key_padding_mask=None, cache=None):
        past_length = 0 if cache is None else cache[0][0].size(2)
        positions = torch.arange(
            past_length, past_length + input_ids.size(1), device=input_ids.device
        )
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        new_cache, all_weights = [], []
        layer_caches = [None] * len(self.blocks) if cache is None else cache
        for block, layer_cache in zip(self.blocks, layer_caches):
            x, layer_cache, weights = block(x, key_padding_mask, layer_cache)
            new_cache.append(layer_cache)
            all_weights.append(weights)
        return self.lm_head(self.final_norm(x)), new_cache, all_weights

model = TinyCausalLM(len(vocabulary)).to(DEVICE)
logits, _, _ = model(input_ids)
print("token IDs:", tuple(input_ids.shape))
print("logits:", tuple(logits.shape))
assert logits.shape == (*input_ids.shape, len(vocabulary))


## 3. Flatten only the example and position axes

Cross-entropy treats each valid token position as one vocabulary classification problem:

```text
logits  (batch, time, vocabulary) -> (batch * time, vocabulary)
labels  (batch, time)             -> (batch * time)
```

The vocabulary axis stays intact. Flattening it too would destroy the candidate-token rows.


In [ ]:
# Canonical loss flattening and a visible one-row alignment
flat_logits = logits.flatten(0, 1)
flat_labels = labels.flatten()
loss = F.cross_entropy(flat_logits, flat_labels, ignore_index=-100)
valid_rows = flat_labels.ne(-100)
print("logits before:", tuple(logits.shape))
print("logits after: ", tuple(flat_logits.shape))
print("targets after:", tuple(flat_labels.shape))
print("valid token lessons:", int(valid_rows.sum()))
print("initial loss:", round(loss.item(), 4))
assert flat_logits.shape[0] == flat_labels.numel()


## 4. One manual loop makes the update boundary visible

The loop deliberately stays native PyTorch. Part 8 will add validation selection, full resume checkpoints, and artifact lineage; this chapter needs only enough training to make generation and cache parity observable.


In [ ]:
# Train the tiny decoder on the Riverside sentence set
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-2, weight_decay=0.01)
losses = []
iterator = iter(loader)
model.train()
for step in range(180):
    try:
        batch = next(iterator)
    except StopIteration:
        iterator = iter(loader)
        batch = next(iterator)
    batch_inputs, batch_labels, batch_padding = [value.to(DEVICE) for value in batch]
    optimizer.zero_grad(set_to_none=True)
    batch_logits, _, _ = model(batch_inputs, batch_padding)
    batch_loss = F.cross_entropy(
        batch_logits.flatten(0, 1),
        batch_labels.flatten(),
        ignore_index=-100,
    )
    batch_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    losses.append(batch_loss.item())

print(f"loss: {losses[0]:.4f} -> {losses[-1]:.4f}")
assert losses[-1] < losses[0]
plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel("optimizer step")
plt.ylabel("next-token loss")
plt.title("Measured training history")
plt.tight_layout()
plt.show()


## 5. Generation needs two independent stopping boundaries

EOS is semantic: the model says the sequence is complete. `max_new_tokens` is operational: the loop stops even if EOS never appears. Keep both.

Temperature `0` below means greedy selection. Positive temperature enables sampling; top-k can restrict its candidate set.


In [ ]:
# Greedy or sampled generation, with either full-prefix recomputation or a KV cache
@torch.no_grad()
def generate(prompt, max_new_tokens=8, temperature=0.0, top_k=None, use_cache=True):
    model.eval()
    generated = [BOS_ID] + [token_to_id[word] for word in prompt.split()]
    cache = None
    next_input = torch.tensor([generated], dtype=torch.long)
    for _ in range(max_new_tokens):
        logits, cache, _ = model(next_input, cache=cache if use_cache else None)
        next_logits = logits[:, -1, :]
        if temperature == 0:
            next_id = int(next_logits.argmax(dim=-1))
        else:
            scaled = next_logits / temperature
            if top_k is not None:
                cutoff = torch.topk(scaled, min(top_k, scaled.size(-1))).values[:, -1:]
                scaled = scaled.masked_fill(scaled < cutoff, float("-inf"))
            next_id = int(torch.multinomial(torch.softmax(scaled, dim=-1), 1))
        generated.append(next_id)
        if next_id == EOS_ID:
            return generated, "eos"
        next_input = torch.tensor([[next_id] if use_cache else generated])
        if not use_cache:
            cache = None
    return generated, "length_limit"

def decode(ids):
    return " ".join(
        id_to_token[index] for index in ids if index not in {PAD_ID, BOS_ID, EOS_ID}
    )

greedy_ids, greedy_stop = generate("aria", temperature=0.0)
torch.manual_seed(SEED)
sampled_ids, sampled_stop = generate("aria", temperature=0.8, top_k=3)
print("greedy:", decode(greedy_ids), "| stop:", greedy_stop)
print("sampled:", decode(sampled_ids), "| stop:", sampled_stop)
assert greedy_stop in {"eos", "length_limit"}
assert sampled_stop in {"eos", "length_limit"}


In [ ]:
# Cache parity: prime the prompt, append one token, and compare with full recomputation
model.eval()
prompt = torch.tensor([[BOS_ID, token_to_id["aria"], token_to_id["heard"]]])
with torch.no_grad():
    full_logits, _, _ = model(prompt)
    prime_logits, prime_cache, _ = model(prompt)
    chosen = prime_logits[:, -1].argmax(dim=-1, keepdim=True)
    cached_next, _, _ = model(chosen, cache=prime_cache)
    extended = torch.cat([prompt, chosen], dim=1)
    recomputed_next, _, _ = model(extended)

prime_delta = (full_logits - prime_logits).abs().max().item()
next_delta = (cached_next[:, -1] - recomputed_next[:, -1]).abs().max().item()
print(f"prompt-logit delta: {prime_delta:.8f}")
print(f"next-logit delta:   {next_delta:.8f}")
assert prime_delta < 1e-6
assert next_delta < 1e-5


## Part 4B scorecard

| Required contract | Executable evidence |
|---|---|
| Dataset/DataLoader/collation | Variable examples became aligned rectangular tensors |
| Mask distinctions | causal, padded-key, and ignored-target boundaries stayed separate |
| Loss flattening | `(B,T,V)` and `(B,T)` became aligned token rows |
| Generation | greedy and sampled loops reported EOS or the hard length limit |
| Cache semantics | cached and recomputed next-token logits matched within tolerance |

**Your turn:** change only `top_k` from `3` to `2`. Predict which candidate disappears before rerunning the sampled generation cell.

**Toy-to-real bridge:** production decoders widen the vectors, stack many blocks, rotate positions with RoPE, group KV heads, and use specialized cache layouts. The contracts measured here do not change.

**Next complaint:** this stack writes on one growing tape. What if a complete source and a separate target need different roles? Continue to [Part 5A](05a-encoder-decoder-and-cross-attention-theory.ipynb).
